# What are components in LlamaIndex?

Alfred is hosting a party and needs to be able to find relevant information on personas that will be attending the party. Therefore, we will use a QueryEngine to index and search through a database of personas.

## Let's install the dependencies
We will install the dependencies for this unit.

In [2]:
!pip install llama-index datasets llama-index-callbacks-arize-phoenix arize-phoenix llama-index-vector-stores-chroma llama-index-llms-huggingface-api llama-index-embeddings-huggingface


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [12]:
!pip install "typer<0.12"

  Attempting uninstall: typer
    Found existing installation: typer 0.24.1
    Uninstalling typer-0.24.1:
      Successfully uninstalled typer-0.24.1

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [13]:
!pip install "click==8.1.8"


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [14]:
!pip install "litellm==1.83.4"


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login, whoami

load_dotenv(dotenv_path="../.env", override=True)
token = os.environ.get("HF_TOKEN")

# Validate before logging in
try:
    login(token=token, add_to_git_credential=False)
    user = whoami()
    print(f"✅ Logged in as: {user['name']}")
except Exception as e:
    print(f"❌ Login failed: {e}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Logged in as: zaidalissa


## Create a QueryEngine for retrieval augmented generation
### Setting up the persona database
We will be using personas from the dvilasuero/finepersonas-v0.1-tiny dataset. This dataset contains 5K personas that will be attending the party!

Let's load the dataset and store it as files in the data directory

In [1]:
from datasets import load_dataset
from pathlib import Path

dataset = load_dataset(path="zaidalissa/finepersonas-v0.1-tiny", split="train")

Path("data").mkdir(parents=True, exist_ok=True)
for i, persona in enumerate(dataset):
    with open(Path("data") / f"persona_{i}.txt", "w") as f:
        f.write(persona["persona"])

### Loading and embedding persona documents
We will use the SimpleDirectoryReader to load the persona descriptions from the data directory. This will return a list of Document objects.

The simplest way to load data is with SimpleDirectoryReader. This versatile component can load various file types from a folder and convert them into Document objects that LlamaIndex can work with. Let’s see how we can use SimpleDirectoryReader to load data from a folder.

In [2]:
from llama_index.core import SimpleDirectoryReader

reader = SimpleDirectoryReader(input_dir="data")
documents = reader.load_data()
len(documents)

5000

#### Some Definitions


* In machine learning, embedding is a representation learning technique that maps complex, high-dimensional data into a lower-dimensional vector space of numerical vectors.
* In practice there are de facto standards: One-Hot, Bag-of-Words, n-grams, TF-IDF, and modern embeddings. Each one makes different tradeoffs. Each one serves different purposes. Understanding them means understanding the natura (the essential nature) of how machines process language. Our objective is make it simple and stupid, which means pick an encoding that preserves enough sensus (feeling) for the task in situ, without wasting compute.

After loading our documents, we need to break them into smaller pieces called Node objects. A Node is just a chunk of text from the original document that’s easier for the AI to work with, while it still has references to the original Document object.

The IngestionPipeline helps us create these nodes through two key transformations.

1. SentenceSplitter breaks down documents into manageable chunks by splitting them at natural sentence boundaries.
2. HuggingFaceEmbedding converts each chunk into numerical embeddings - vector representations that capture the semantic meaning in a way AI can process efficiently.
This process helps us organise our documents in a way that’s more useful for searching and analysis.

In [3]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline

# create the pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
    ]
)

# run the pipeline sync or async
nodes = await pipeline.arun(documents=documents[:10])
nodes[0]

TextNode(id_='9380ace2-48d8-4d8d-80ac-8427f3a99994', embedding=[-0.05962567403912544, 0.02040313556790352, 0.02128954418003559, -0.010687166824936867, -0.012180567719042301, -0.04411396384239197, 0.007198816165328026, 0.015415969304740429, -0.04678264632821083, -0.0436910018324852, -0.0007798787555657327, -0.03973851725459099, -0.019043656066060066, 0.024635804817080498, -0.005054472479969263, -0.036002498120069504, -0.007778162136673927, 0.04452042654156685, 0.011408746242523193, 0.007581743877381086, -0.0075704301707446575, -0.07013014703989029, 0.06867500394582748, -0.03700421378016472, -0.04977079853415489, 0.04120711237192154, 0.03824923187494278, -0.006207672879099846, -0.0028668311424553394, -0.15109239518642426, -0.019949719309806824, 0.0013386401114985347, -0.018280742689967155, 0.014136574231088161, 0.08616043627262115, 0.03856253996491432, 0.007805913686752319, 0.06434840708971024, -0.010406319051980972, 0.010279250331223011, 0.028873905539512634, 0.012602943927049637, -0.04

In [4]:
nodes[0].metadata

{'file_path': '/Users/clean_zaid/Documents/Home/Backups/BOOKS/Trillionaire/Python/jupyter-notebooks/huggingface-ai-agents-course/unit_2.2/data/persona_0.txt',
 'file_name': 'persona_0.txt',
 'file_type': 'text/plain',
 'file_size': 132,
 'creation_date': '2026-04-09',
 'last_modified_date': '2026-04-10'}

In [5]:
nodes[0].text

'A local art historian and museum professional interested in 19th-century American art and the local cultural heritage of Cincinnati.'

In [6]:
print(nodes[0].embedding)

[-0.05962567403912544, 0.02040313556790352, 0.02128954418003559, -0.010687166824936867, -0.012180567719042301, -0.04411396384239197, 0.007198816165328026, 0.015415969304740429, -0.04678264632821083, -0.0436910018324852, -0.0007798787555657327, -0.03973851725459099, -0.019043656066060066, 0.024635804817080498, -0.005054472479969263, -0.036002498120069504, -0.007778162136673927, 0.04452042654156685, 0.011408746242523193, 0.007581743877381086, -0.0075704301707446575, -0.07013014703989029, 0.06867500394582748, -0.03700421378016472, -0.04977079853415489, 0.04120711237192154, 0.03824923187494278, -0.006207672879099846, -0.0028668311424553394, -0.15109239518642426, -0.019949719309806824, 0.0013386401114985347, -0.018280742689967155, 0.014136574231088161, 0.08616043627262115, 0.03856253996491432, 0.007805913686752319, 0.06434840708971024, -0.010406319051980972, 0.010279250331223011, 0.028873905539512634, 0.012602943927049637, -0.04036947339773178, 0.05734625086188316, 0.0996483564376831, 0.026

As, you can see, we have created a list of Node objects, which are just chunks of text from the original documents. Let's explore how we can add these nodes to a vector store.

### Storing and indexing documents

After creating our Node objects we need to index them to make them searchable, but before we can do that, we need a place to store our data.

Since we are using an ingestion pipeline, we can directly attach a vector store to the pipeline to populate it. In this case, we will use Chroma to store our documents.

In [7]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore

db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection(name="alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(),
        HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5"),
    ],
    vector_store=vector_store,
)

nodes = await pipeline.arun(documents=documents[:10])
len(nodes)

10

This is where vector embeddings come in - by embedding both the query and nodes in the same vector space, we can find relevant matches. The VectorStoreIndex handles this for us, using the same embedding model we used during ingestion to ensure consistency.

---


**The problem it's solving**

When you ask a question like *"What's a good dinner menu for 8 people?"*, the system needs to find relevant chunks of text (nodes) stored in the database. But you can't just do a keyword search — the question might be worded completely differently from the stored text.

**What are vector embeddings?**

An embedding model converts text into a list of numbers (a vector) that represents its *meaning*. For example:

- *"dinner party menu"* → `[0.12, 0.87, 0.34, ...]`
- *"meal plan for guests"* → `[0.11, 0.85, 0.36, ...]` ← similar numbers, similar meaning
- *"car engine repair"* → `[0.91, 0.02, 0.77, ...]` ← very different numbers

**"Embedding both the query and nodes in the same vector space"**

This means:
- When you **store** your documents, each chunk (node) gets converted to a vector
- When you **search**, your question also gets converted to a vector using the **same model**
- Then the system finds nodes whose vectors are closest to the query vector — those are the relevant matches

**Why "same embedding model" matters**

If you embedded your documents with model A but searched with model B, the numbers would be on completely different scales — like measuring a room in feet then comparing it to a measurement taken in meters. Using the same model (`BAAI/bge-small-en-v1.5` in the code) ensures both the stored nodes and your query live in the same numerical space, so the comparison is meaningful.

**What `VectorStoreIndex` does for you**

It handles all of this automatically — you just pass it the vector store and the embedding model, and it takes care of embedding your queries and finding the closest matching nodes at search time.

---

Our example: it's a naming thing that trips people up.

**The query** = your question at search time
> e.g. *"What's a good dinner menu for 8 people?"*

**The nodes** = the chunks of text stored from your documents during ingestion
> e.g. a paragraph from a PDF about recipes, split up by `SentenceSplitter`

So the flow is:

1. **Earlier (ingestion):** your documents got split into nodes → each node got embedded and stored in ChromaDB
2. **Now (search):** you type a question (the query) → it also gets embedded → the system compares it against all the stored node vectors → returns the closest ones

In the code from the page, the nodes were created here:
```python
nodes = await pipeline.arun(documents=[Document.example()])
```

And the query happens here:
```python
query_engine.query("What is the meaning of life?")
```

That string `"What is the meaning of life?"` — **that's the query**. Everything stored before that moment are the **nodes**.

---

Let’s see how to create this index from our vector store and embeddings:

In [8]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)

All information is automatically persisted within the ChromaVectorStore object and the passed directory path.

Great! Now that we can save and load our index easily, let’s explore how to query it in different ways.

### Querying a VectorStoreIndex with prompts and LLMs
Before we can query our index, we need to convert it to a query interface. The most common conversion options are:

1. as_retriever: For basic document retrieval, returning a list of NodeWithScore objects with similarity scores
2. as_query_engine: For single question-answer interactions, returning a written response
3. as_chat_engine: For conversational interactions that maintain memory across multiple messages, returning a written response using chat history and indexed context

We’ll focus on the query engine since it is more common for agent-like interactions. We also pass in an LLM to the query engine to use for the response.

In [9]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")
query_engine = index.as_query_engine(
    llm=llm,
    response_mode="tree_summarize",
)
response = query_engine.query("What is the meaning of life?")
# The meaning of life is 42
response

Response(response="The meaning of life is a profound and complex question that varies widely across different cultures, histories, and societies. For someone deeply immersed in Cypriot culture, the meaning of life might be intertwined with the island's rich heritage, its resilient people, and the importance of community and family. It could also reflect the island's geographical position as a bridge between East and West, influencing perspectives on identity, tradition, and modernity.", source_nodes=[NodeWithScore(node=TextNode(id_='cd4883f0-cb15-4990-b4d1-1aa6e711510f', embedding=None, metadata={'file_path': '/Users/clean_zaid/Documents/Home/Backups/BOOKS/Trillionaire/Python/jupyter-notebooks/huggingface-ai-agents-course/unit_2.2/data/persona_1.txt', 'file_name': 'persona_1.txt', 'file_type': 'text/plain', 'file_size': 266, 'creation_date': '2026-04-09', 'last_modified_date': '2026-04-09'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_mod

#### Explanation:
Let me use your actual output to answer everything.

---

**From where did the agent get the response?**

From two `.txt` files on your machine:
- `persona_1.txt` — about an anthropologist studying Cypriot culture
- `persona_1001.txt` — about a web developer/student

Those are your **nodes** (the stored chunks).

---

**What are the steps to get the response?**

**Step 1 — Query comes in:**
> *"What is the meaning of life?"*

**Step 2 — Query gets embedded** into a vector using `BAAI/bge-small-en-v1.5`

**Step 3 — Vector search** compares it against all stored node vectors and returns the 2 closest matches:
- `persona_1.txt` → score `0.387`
- `persona_1001.txt` → score `0.370`

**Step 4 — ResponseSynthesizer** takes those 2 nodes + your question and sends them to the LLM

**Step 5 — LLM generates the final answer** using the node content as context

---

**How was this specific response generated?**

The LLM received something like:
> *"Given this context: [anthropologist text] + [web developer text], answer: what is the meaning of life?"*

So it blended both personas into the answer — that's why the response says:
- *"for an anthropologist... societal norms"* ← from `persona_1.txt`
- *"for a web developer... digital experiences"* ← from `persona_1001.txt`

---

**Which ResponseSynthesizer strategy was used?**

You used `tree_summarize` (you set it explicitly in the code):
```python
query_engine = index.as_query_engine(
    llm=llm,
    response_mode="tree_summarize",
)
```

So it went through both nodes and built a synthesized, structured answer combining them — rather than just picking one or refining sequentially.

#### Second Example

In [10]:
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
import nest_asyncio

nest_asyncio.apply()  # This is needed to run the query engine
llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")
query_engine = index.as_query_engine(
    llm=llm,
    response_mode="tree_summarize",
)
response2 = query_engine.query(
    "Respond using a persona that describes author and travel experiences?"
)
response2

Response(response='This individual is deeply engaged in the study of Cypriot culture, history, and society. With a significant amount of time dedicated to research and living in Cyprus, they have cultivated a profound understanding of the local people, their customs, and the broader way of life on the island.', source_nodes=[NodeWithScore(node=TextNode(id_='2a91e649-68f3-4424-8b18-484b995e1763', embedding=None, metadata={'file_path': '/Users/clean_zaid/Documents/Home/Backups/BOOKS/Trillionaire/Python/jupyter-notebooks/huggingface-ai-agents-course/unit_2.2/data/persona_1.txt', 'file_name': 'persona_1.txt', 'file_type': 'text/plain', 'file_size': 266, 'creation_date': '2026-04-09', 'last_modified_date': '2026-04-09'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships=

In [11]:
response2.source_nodes[0].metadata['file_name'], response2.source_nodes[0].score

('persona_1.txt', 0.5178895636466943)

### Response Processing
Under the hood, the query engine doesn’t only use the LLM to answer the question but also uses a ResponseSynthesizer as a strategy to process the response. Once again, this is fully customisable but there are three main strategies that work well out of the box:

* refine: create and refine an answer by sequentially going through each retrieved text chunk. This makes a separate LLM call per Node/retrieved chunk.
* compact (default): similar to refining but concatenating the chunks beforehand, resulting in fewer LLM calls.
* tree_summarize: create a detailed answer by going through each retrieved text chunk and creating a tree structure of the answer.

The language model won’t always perform in predictable ways, so we can’t be sure that the answer we get is always correct. We can deal with this by evaluating the quality of the answer.

## Evaluation and observability
LlamaIndex provides built-in evaluation tools to assess response quality. These evaluators leverage LLMs to analyze responses across different dimensions. Let’s look at the three main evaluators available:

* FaithfulnessEvaluator: Evaluates the faithfulness of the answer by checking if the answer is supported by the context.
* AnswerRelevancyEvaluator: Evaluate the relevance of the answer by checking if the answer is relevant to the question.
* CorrectnessEvaluator: Evaluate the correctness of the answer by checking if the answer is correct.


In [12]:
from llama_index.core.evaluation import FaithfulnessEvaluator

# query index
evaluator = FaithfulnessEvaluator(llm=llm)
eval_result = evaluator.evaluate_response(response=response)
eval_result.passing

False

In [13]:
from llama_index.core.evaluation import FaithfulnessEvaluator

# query index
evaluator = FaithfulnessEvaluator(llm=llm)
eval_result = evaluator.evaluate_response(response=response2)
eval_result.passing

True

In [14]:
from llama_index.core.evaluation import FaithfulnessEvaluator

# query index
evaluator = FaithfulnessEvaluator(llm=llm)
response = query_engine.query(
    "What battles took place in New York City in the American Revolution?"
)
eval_result = evaluator.evaluate_response(response=response)
eval_result.passing

False

Even without direct evaluation, we can gain insights into how our system is performing through observability. This is especially useful when we are building more complex workflows and want to understand how each component is performing.

In [27]:
!pip install openinference-instrumentation-llama-index arize-otel


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [11]:
!pip install --upgrade arize-otel


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
# tracer_provider = register(
#     space_id=os.getenv("ARIZE_SPACE_ID"),
#     api_key=os.getenv("ARIZE_API_KEY"),
#     project_name="hf-llamaindex-app"
# )

import os
from arize.otel import register
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

tracer_provider = register(
 space_id=os.getenv("ARIZE_SPACE_ID"),
    api_key=os.getenv("ARIZE_API_KEY"),
    project_name="my-llamaindex-app",
    batch=False,
    log_to_console=True
)

LlamaIndexInstrumentor().instrument(tracer_provider=tracer_provider,skip_dep_check=True)

🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: my-llamaindex-app
|  Span Processor: Multiple Span Processors
|  Collector Endpoint: Multiple Span Exporters
|  Transport: Multiple Span Exporters
|  Transport Headers: Multiple Span Exporters
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [10]:
!pip show arize-otel

Name: arize-otel
Version: 0.12.0
Summary: Helper package for OTEL setup to send traces to Arize & Phoenix
Home-page: 
Author: 
Author-email: Arize AI <support@arize.com>
License: BSD
Location: /Users/clean_zaid/Documents/Home/Backups/BOOKS/Trillionaire/Python/jupyter-notebooks/.venv/lib/python3.12/site-packages
Requires: openinference-instrumentation, openinference-semantic-conventions, opentelemetry-exporter-otlp, opentelemetry-proto, opentelemetry-sdk
Required-by: 


In [15]:
response = query_engine.query("What is the meaning of life?")
print(response)

The meaning of life is a profound and complex question that varies widely across different cultures, histories, and societies. For someone deeply immersed in Cypriot culture, the meaning of life might be intertwined with family values, community bonds, and the rich heritage of the island. It could also reflect the importance of resilience, given the island's historical experiences, or the pursuit of simplicity and connection with nature, which are often valued aspects of Cypriot life.


In [16]:
response = query_engine.query("What is the meaning of life?")
print(response)

# Force flush all buffered spans
tracer_provider.force_flush()

The meaning of life is a profound and complex question that varies widely across different cultures, histories, and societies. It encompasses philosophical, religious, and personal perspectives, each offering unique insights into what gives existence purpose or significance. For someone deeply immersed in Cypriot culture, the meaning of life might be intertwined with the island's rich history, its traditions, and the values that its people hold dear, such as family, community, and resilience in the face of challenges.


NameError: name 'tracer_provider' is not defined